In [2]:
import warnings 
warnings.filterwarnings('ignore')
import langchain_community
from langchain_community.document_loaders import PyPDFLoader
loder = PyPDFLoader('C:\\Users\\SAYAN METE\\OneDrive\\Documents\\Retrival Argumented Generation\\hybrid_RAG\\Virat_Kohli_Complete_With_Personal_Life.pdf')
pages = loder.load()


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
spliter = RecursiveCharacterTextSplitter(chunk_size = 2000,chunk_overlap = 200)
text = spliter.split_documents(pages)

chunk = []
for i in text:
    chunk.append(i.page_content)
metadata = []
for i in text:
    metadata.append(i.metadata)

import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
embedding_function = SentenceTransformerEmbeddingFunction()

client = chromadb.PersistentClient(path="./database-3")
collection = client.get_or_create_collection(name="Collection",embedding_function=embedding_function)

try:
    if collection.count() == 0:
        collection.add(
            documents=chunk,
            ids = [str(i) for i in range(len(chunk))],
            metadatas=metadata
        )

except Exception as e:
    print(str(e))


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7932.23it/s]


In [1]:
from rank_bm25 import BM25Okapi

def token_create(i):
    i = i.lower()
    i = i.split()
    return i
token = [token_create(i) for i in chunk]
token_for_keyword_search = BM25Okapi(token)


NameError: name 'chunk' is not defined

In [4]:
def retrival(query:str)->str:
    query_lower_case = query.lower()

    response = collection.query(query_texts=[query_lower_case],n_results=5)
    document = response['documents'][0]
    distance = response['distances'][0]

    thresold = 1.6
    near_chunk = []
    for i , j in zip(distance,document):
        if thresold>i:
            near_chunk.append(j)
    score = token_for_keyword_search.get_scores(token_create(query_lower_case))

    def near_index_find(score,k=10):
        index = list(enumerate(score))
        index_sorted = sorted(index, key=lambda x:x[1],reverse=True)
        return [inx for inx , sc in index_sorted[:k]]

    get_index = near_index_find(score,k = 10)
    index_to_chunk = []
    for i in get_index:
        index_to_chunk.append(chunk[i])

    rrf_item = {}

    for rank , doc in enumerate(near_chunk):
        rrf_item[doc] = rrf_item.get(doc,0)+1/(rank+60)
    for rank , doc in enumerate(index_to_chunk):
        rrf_item[doc] = rrf_item.get(doc,0)+1/(rank+60)

    merge = sorted(rrf_item.items(),key=lambda x:x[1],reverse=True)

    top_related_doc = []
    for doc,_ in merge:
        top_related_doc.append(doc)
    if not top_related_doc:
        return "NOT RELATED CONTENT"
    return "\n\n".join(top_related_doc)


    

In [5]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()
api = os.getenv('GROQ_API_KEY') 
groq_llm_model = ChatGroq(model='openai/gpt-oss-120b',api_key=api)

question = input('ask your question about virat kholi??')
content = retrival(question)

prompt = """
You are a realiable ai assistent so provide user asking qustions based on only the local document 

content = {content}
qustion = {question}
"""
final_prompt = prompt.format(content=content,question=question)
print(groq_llm_model.invoke(final_prompt).content)


**Virat Kohli** is an Indian international cricketer, widely regarded as one of the leading batters of the modern era.

- **Full name:** Virat Kohli  
- **Born:** 5 November 1988, Delhi, India  
- **Role:** Right‑handed batter (also bowls right‑arm medium)  
- **International debut:** ODI, 2008 (against Sri Lanka)  
- **Domestic/IPL team:** Royal Challengers Bengaluru (RCB)  
- **Key achievements:**  
  - Captain of India’s 2008 Under‑19 World Cup‑winning side  
  - Major contributor to India’s 2011 ODI World Cup, 2013 Champions Trophy, 2024 T20 World Cup and 2025 Champions Trophy victories  
  - Holds 123 Test matches (9,230 runs, 30 centuries) and 314 ODIs (14,941 runs, 54 centuries) as of September 2026  
  - First player to reach 50 ODI centuries; ICC Men’s ODI Cricketer of the Decade and ICC Men’s Cricketer of the Decade (2020)  

**Personal life (publicly reported):**  
- Married actress Anushka Sharma in December 2017; they have two children, daughter Vamika (born 2021) and son 